In [ ]:
import pandas as pd

# Check Composition

## Sample All

In [ ]:
all_data_sample = pd.read_parquet('../data/us_10m_nointernship_2018_2024_benefits.parquet.gzip')

In [ ]:
salary_data_sample = pd.read_parquet('../data/small_samples/2024_salary_sample.parquet.gzip')

In [ ]:
industry_crosswalk = pd.read_csv('../../thesis/Resources/Expanded_NAICS_Code_to_Industry_Name_Crosswalk.csv')

In [ ]:
bls_stats = pd.read_csv('../../thesis/Resources/Employment_by_Industry.csv')

In [ ]:
industry_crosswalk = industry_crosswalk.rename(columns = {'Industry name': 'BLS_industry'})

In [ ]:
industry_crosswalk.dropna(inplace=True)

In [ ]:
industry_crosswalk['NAICS code'] = industry_crosswalk['NAICS code'].astype(int)

In [ ]:
industry_crosswalk = industry_crosswalk.rename(columns = {'NAICS code': 'BLS_NAICS'})

In [ ]:
all_data_sample = all_data_sample.merge(industry_crosswalk, left_on = 'NAICS_2022_2', right_on = 'BLS_NAICS', how = 'left')

In [ ]:
all_data_sample

In [ ]:
all_data_sample.loc[all_data_sample['BLS_industry'].isna(), 'BLS_industry'] = all_data_sample.loc[all_data_sample['BLS_industry'].isna(), 'NAICS_2022_2_NAME']

In [ ]:
all_data_sample.groupby('BLS_industry', dropna=False).size()

In [ ]:
def get_grouped_industry(data):
    grouped_industry = data.groupby('BLS_industry', dropna=False).size().reset_index(name='count')

    # Calculate the total count
    total_count = grouped_industry['count'].sum()

    # Calculate the percentage of each group
    grouped_industry['percentage'] = (grouped_industry['count'] / total_count) * 100

    return grouped_industry


In [ ]:
grouped_industry = get_grouped_industry(all_data_sample)

In [ ]:
industry_comp = grouped_industry.merge(bls_stats, left_on = 'BLS_industry', right_on = 'Industry name', how = 'outer')

In [ ]:
industry_comp = industry_comp[['BLS_industry', 'Industry name','percentage', 'Percent distribution 2022']]

In [ ]:
industry_comp.rename(columns = {'percentage': 'OJV Data', 'Percent distribution 2022': 'National Statistics'}, inplace=True)

In [ ]:
industry_comp

In [ ]:
import matplotlib.pyplot as plt

# Filter out rows with missing values in either column
filtered_df = industry_comp.dropna(subset=['OJV Data', 'National Statistics'])

# Create a scatter plot
plt.figure(figsize=(12, 8))
plt.scatter(filtered_df['OJV Data'], filtered_df['National Statistics'], color='blue')

# Add labels for each point
for i, row in filtered_df.iterrows():
    plt.text(row['OJV Data'], row['National Statistics'], row['BLS_industry'], fontsize=9, ha='right')

# Set the labels and title
plt.xlabel('OJV Data')
plt.ylabel('National Statistics')
max_val = max(filtered_df['OJV Data'].max(), filtered_df['National Statistics'].max()) * 1

plt.plot([-0.5, max_val], [-0.5, max_val], linestyle='dotted', color='red')

# plt.title('Percentage vs National Statistics by BLS Industry')
plt.axis('equal')
# plt.xlim(0, max(filtered_df['OJV Data'].max(), filtered_df['National Statistics'].max()) * 1.1)
# plt.ylim(0, max(filtered_df['OJV Data'].max(), filtered_df['National Statistics'].max()) * 1.1)

# Show the plot
# Show the plot
plt.grid(True)
# save plot
plt.savefig('../figures/industry composition check/industry_comp_2024.png')
plt.show()


# Salary Sample

In [ ]:
salary_data_sample = salary_data_sample.merge(industry_crosswalk, left_on = 'NAICS_2022_2', right_on = 'BLS_NAICS', how = 'left')

In [ ]:
salary_data_sample.loc[salary_data_sample['BLS_industry'].isna(), 'BLS_industry'] = salary_data_sample.loc[salary_data_sample['BLS_industry'].isna(), 'NAICS_2022_2_NAME']

In [ ]:
salary_data_sample.groupby('BLS_industry', dropna=False).size()

In [ ]:
# salary_data_sample.to_parquet('../data/salary_sample_body.parquet.gzip', compression='gzip')

In [ ]:
grouped_industry_salary = get_grouped_industry(salary_data_sample)

In [ ]:
industry_comp_salary = grouped_industry_salary.merge(bls_stats, left_on = 'BLS_industry', right_on = 'Industry name', how = 'outer')

In [ ]:
industry_comp_salary = industry_comp_salary[['BLS_industry', 'Industry name','percentage', 'Percent distribution 2022']]

In [ ]:
industry_comp_salary.rename(columns = {'percentage': 'OJV Data', 'Percent distribution 2022': 'National Statistics'}, inplace=True)

In [ ]:
industry_comp_salary

In [ ]:
import matplotlib.pyplot as plt

# Filter out rows with missing values in either column
filtered_df_salary = industry_comp_salary.dropna(subset=['OJV Data', 'National Statistics'])

# Create a scatter plot
plt.figure(figsize=(12, 8))
plt.scatter(filtered_df_salary['OJV Data'], filtered_df_salary['National Statistics'], color='blue')

# Add labels for each point
for i, row in filtered_df_salary.iterrows():
    plt.text(row['OJV Data'], row['National Statistics'], row['BLS_industry'], fontsize=9, ha='right')

# Set the labels and title
plt.xlabel('OJV Data')
plt.ylabel('National Statistics')
max_val = max(filtered_df_salary['OJV Data'].max(), filtered_df_salary['National Statistics'].max()) * 1

plt.plot([-0.5, max_val], [-0.5, max_val], linestyle='dotted', color='red')

# plt.title('Percentage vs National Statistics by BLS Industry')
plt.axis('equal')
# plt.xlim(0, max(filtered_df_salary['OJV Data'].max(), filtered_df_salary['National Statistics'].max()) * 1.1)
# plt.ylim(0, max(filtered_df_salary['OJV Data'].max(), filtered_df_salary['National Statistics'].max()) * 1.1)

# Show the plot
# Show the plot
plt.grid(True)
# save plot
plt.savefig('../figures/industry composition check/ndustry_comp_2024_samplesalary.png')
plt.show()
